In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import re
import time
from datetime import datetime
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from urllib.request import urlopen
from bs4 import BeautifulSoup
import requests
import urllib.parse
from time import sleep

In [51]:
from random import randint


root_url = 'https://www.amazon.com/'
week = '2026-02-01'

def get_data(date):  
    #headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}
    headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

    r = requests.get('https://www.amazon.com/charts/'+ date +'/mostsold/fiction', headers=headers)
    content = r.content
    soup = BeautifulSoup(content)

# //*[@id="sin-body"]/div/div/div/div[2]
    testsoup = soup.find('div', attrs={'class': 'kc-vertical-rank-container row'})
    all = []

    if testsoup is not None:
        list_items = testsoup.find_all('div')

        for book in list_items:
            #create tags for each element we want
            ranktag = book.find('div', attrs={'class': 'kc-rank-card-rank'}) if book.find('div', attrs={'class': 'kc-rank-card-rank'}) else None

            authtag = book.find('div', attrs={'class': 'kc-rank-card-author'}) if book.find('div', attrs={'class': 'kc-rank-card-author'}) else None

            titletag = book.find('div', attrs={'class': 'kc-rank-card-title'}) if book.find('div', attrs={'class': 'kc-rank-card-title'}) else None

            hrefs = book.find('div', attrs={'class': 'kc-book-title-img'}).find_all('a') if book.find('div', attrs={'class': 'kc-book-title-img'}) else None
            href = hrefs[0] if hrefs else None

            if href is not None:
                book_descr = href.get('href')

                #join urls to go to book page
                book_descr_url = root_url + book_descr

                try:
                    book_page = requests.get(book_descr_url, headers=headers)
                    book_soup = BeautifulSoup(book_page.content)

                    #find blurb and extract text
                    blurb_tag = book_soup.find('div', attrs={'id': 'bookDescription_feature_div'})
                    blurb = blurb_tag.get_text(strip=False) if blurb_tag else None

                    #find publisher and extract text
                    details = book_soup.find('ul', attrs={'class': 'a-unordered-list a-nostyle a-vertical a-spacing-none detail-bullet-list'})
                    details_list = details.find_all('li') if details else None
                    pub = details_list[1] if details_list else None
                    pub = pub.find('span', attrs={'class': 'a-text-bold'}).find_next_sibling('span').get_text(strip=True) if pub and pub.find('span', attrs={'class': 'a-text-bold'}) else None

                    #pub date
                    pub_date = details_list[3].find('span', attrs={'class': 'a-text-bold'}).find_next_sibling('span').get_text(strip=True) if details_list else None

                    #series info
                    series_info = details_list[13].find_all('span') if details_list else None
                    if series_info is not None and len(series_info) == 3:
                        series = 1
                        series_name = series_info[2].get_text(strip=True)
                        series_order = series_info[1].get_text(strip=True)[:15]
                    else:
                        series = 0
                        series_name = None
                        series_order = None

                    #genre info
                    genre_tag = book_soup.find('ul', attrs={'class': 'a-unordered-list a-horizontal a-size-small'}).find_all('li')
                    genres = [tag.get_text(strip=True) for tag in genre_tag if len(tag.get_text(strip=True)) > 2]

                    #format
                    if 'Kindle Store' in genres:
                        if 'Kindle eBooks' in genres:
                            format = 'eBook'
                        elif 'Kindle Audiobooks' in genres:
                            format = 'Audiobook'
                        genres = genres[1:]
                    elif 'Paperback' in genres:
                        format = 'Paperback'
                    elif 'Hardcover' in genres:
                        format = 'Hardcover'
                    elif 'Book' in genres:
                        format = 'Book'
                    else:
                        format = None
                    genres = genres[1:]
                except:
                    blurb = None
                    pub = None
                    pub_date = None
                    series = 0
                    series_name = None
                    series_order = None
                    genres = []
                    format = None
                
            
            if ranktag is not None and authtag is not None and titletag is not None and href is not None:
                    all.append({
                            'week': date,
                            'rank': ranktag.get_text(strip=True) if ranktag else None,
                            'author': authtag.get_text(strip=True)[3::] if authtag else None,
                            'title': titletag.get_text(strip=True) if titletag else None,
                            #'href': href.get('href') if href else None,
                            'blurb': blurb,
                            'publisher': pub,
                            'publication_date': pub_date,
                            'series': series,
                            'series_name': series_name,
                            'series_order': series_order,
                            'genres': genres,
                            'format': format,
                    })
        sleep(randint(1,4))

    #<a class="kc-cover-link app-specific-display not_app" href="/dp/B0FJTF5MJB/ref=chrt_bk_sd_fc_1_ci_lp"> <img alt="Cover image of Dear Debbie by Freida McFadden" src="https://m.media-amazon.com/images/I/81K9E2AyfjL.jpg" title="Cover image of Dear Debbie by Freida McFadden"/>
    return all

In [52]:
get_data(week)

[{'week': '2026-02-01',
  'rank': '1',
  'author': 'Freida McFadden',
  'title': 'Dear Debbie',
  'blurb': '\n  Sometimes, enough is enough…Debbie Mullen is losing it. For years, she has compiled all of her best advice into her column, Dear Debbie, where the wives of New England come for sympathy and neighborly advice. Through her work, Debbie has heard from countless women who are ignored, belittled, or even abused by their husbands. And Debbie does her best to guide them in the right direction. Or at least, she did.These days, Debbie’s life seems to be spiraling out of control. She just lost her job. Something strange is happening with her teenage daughters. And her husband is keeping secrets, according to the tracking app she installed on his phone. Now, Debbie’s done being the bigger person.She’s done being reasonable and practical. It’s time to take her own advice.And now it’s time for payback against all the people in her life who deserve it the most.From #1 New York Times and in

In [99]:
df_test = pd.DataFrame(get_data(week))
df_test.tail()

,week,rank,author,title,blurb,publisher
15,2026-02-01,16,Freida McFadden,The Housemaid Is Watching,\n “You must be our new neighbors!” Mrs. Lowe...,Bookouture
16,2026-02-01,17,Alice Schertle,Little Blue Truck's Valentine,\n Spread the love with Little Blue Truck—a p...,"December 8, 2020"
17,2026-02-01,18,Rachel Reid,Game Changer,\n THE SERIES THAT INSPIRED HEATED RIVALRY • ...,Carina Press
18,2026-02-01,19,John Grisham,The Widow,\n #1 NEW YORK TIMES BESTSELLER • John Grisha...,Doubleday
19,2026-02-01,20,Julia Quinn,An Offer from a Gentleman,\n The inspiration for season four of BRIDGER...,Avon


In [23]:
#FOR TESTING

# if href is not None:
#book_descr = href.get('href')
headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}
book_descr_url = 'https://www.amazon.com/dp/B09DW1YSGQ/ref=chrt_bk_sd_fc_19_ci_lp'

book_page = requests.get(book_descr_url, headers=headers)
book_soup = BeautifulSoup(book_page.content)
book_soup
blurb_tag = book_soup.find('div', attrs={'id': 'bookDescription_feature_div'})
blurb = blurb_tag.get_text(strip=True) if blurb_tag else None
blurb
details = book_soup.find('ul', attrs={'class': 'a-unordered-list a-nostyle a-vertical a-spacing-none detail-bullet-list'})
details_list = details.find_all('li')
pub = details_list[1] if details else None
pub = pub.find('span', attrs={'class': 'a-text-bold'}).find_next_sibling('span').get_text(strip=True) if pub and pub.find('span', attrs={'class': 'a-text-bold'}) else None
#.find_next_sibling('li').find('span', attrs={'class': 'a-text-bold'}).find_next_sibling('span') if pub_tag else None
pub
date = details_list[3].find('span', attrs={'class': 'a-text-bold'}).find_next_sibling('span').get_text(strip=True) if details_list[3] and details_list[3].find('span', attrs={'class': 'a-text-bold'}) else None
date

series_info = details_list[13].find_all('span')
series_place = series_info[1].get_text(strip=True)
series_place
#Trying to find if it is in a series
#

'Book 6 of 7\n                                            \u200f\n                                            :\n                                            \u200e'

In [107]:
len(details_list[13].find_all('span'))

3

In [48]:

headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}
r = requests.get('https://www.amazon.com/charts/2026-02-01/mostsold/fiction?ref=chrt_bk_nav_fwd', headers=headers)
content = r.content
soup = BeautifulSoup(content, 'html.parser')
# //*[@id="sin-body"]/div/div/div/div[2]
testsoup = soup.find('div', attrs={'class': 'kc-vertical-rank-container row'})
all = []

# if testsoup is not None:
#     list_items = testsoup.find_all('div')
#     for book in list_items:
#             #create tags for each element we want
#         ranktag = book.find('div', attrs={'class': 'kc-rank-card-rank'})

#         authtag = book.find('div', attrs={'class': 'kc-rank-card-author'})

#         titletag = book.find('div', attrs={'class': 'kc-rank-card-title'})

#         hrefs = book.find('div', attrs={'class': "kc-book-title-img-section"})

testing = testsoup.find_all('div')
testbook = testing[7]
for ind, obj in enumerate(testing):
    print(ind, obj.get_text(strip = True))


0 1First week on the listDear Debbieby Freida McFaddenRead a Kindle Book sample of Dear DebbieListen to an audiobook preview of Dear DebbieCUSTOMER REVIEWS4.3 / 36,615 REVIEWS
1 1
2 1
3 1
4 
5 
6 First week on the listDear Debbieby Freida McFaddenRead a Kindle Book sample of Dear DebbieListen to an audiobook preview of Dear Debbie
7 First week on the listDear Debbieby Freida McFadden
8 First week on the list
9 Dear Debbie
10 by Freida McFadden
11 
12 
13 Listen to an audiobook preview of Dear Debbie
14 Listen to an audiobook preview of Dear Debbie
15 
16 
17 
18 
19 
20 
21 
22 
23 
24 
25 CUSTOMER REVIEWS4.3 / 36,615 REVIEWS
26 CUSTOMER REVIEWS4.3 / 36,615 REVIEWS
27 CUSTOMER REVIEWS4.3 / 36,615 REVIEWS
28 CUSTOMER REVIEWS
29 
30 4.3 / 36,615 REVIEWS
31 
32 4.3 / 36,615 REVIEWS
33 29 weeks on the listTheo of Goldenby Allen LeviRead a Kindle Book sample of Theo of GoldenListen to an audiobook preview of Theo of GoldenCUSTOMER REVIEWS4.7 / 44,792 REVIEWS
34 2
35 2
36 2
37 
38 
39 9 week

In [85]:
from datetime import date, timedelta

def all_sundays_of_year(year):
    """
    Generates all the dates for Sundays in a given year.
    """
    d = date(year, 1, 1) # Start with January 1st
    # Adjust d to the first Sunday of the year. 
    # weekday() returns 0 for Monday, 6 for Sunday.
    # The calculation d.weekday() gives days to add to reach Sunday.
    d += timedelta(days=6 - d.weekday())
    
    while d.year == year:
        yield d
        d += timedelta(days=7) # Move to the next Sunday

# Example usage for the year 2024:
year_to_check = 2024
sundays_2024 = list(all_sundays_of_year(year_to_check))
sundays = []
for sunday in sundays_2024:
    sundays.append(sunday.strftime("%Y-%m-%d"))

# print(f"All Sundays of the year {year_to_check}:")
# for sunday in sundays_2024:
#     print(sunday.strftime("%Y-%m-%d"))
sundays

['2024-01-07',
 '2024-01-14',
 '2024-01-21',
 '2024-01-28',
 '2024-02-04',
 '2024-02-11',
 '2024-02-18',
 '2024-02-25',
 '2024-03-03',
 '2024-03-10',
 '2024-03-17',
 '2024-03-24',
 '2024-03-31',
 '2024-04-07',
 '2024-04-14',
 '2024-04-21',
 '2024-04-28',
 '2024-05-05',
 '2024-05-12',
 '2024-05-19',
 '2024-05-26',
 '2024-06-02',
 '2024-06-09',
 '2024-06-16',
 '2024-06-23',
 '2024-06-30',
 '2024-07-07',
 '2024-07-14',
 '2024-07-21',
 '2024-07-28',
 '2024-08-04',
 '2024-08-11',
 '2024-08-18',
 '2024-08-25',
 '2024-09-01',
 '2024-09-08',
 '2024-09-15',
 '2024-09-22',
 '2024-09-29',
 '2024-10-06',
 '2024-10-13',
 '2024-10-20',
 '2024-10-27',
 '2024-11-03',
 '2024-11-10',
 '2024-11-17',
 '2024-11-24',
 '2024-12-01',
 '2024-12-08',
 '2024-12-15',
 '2024-12-22',
 '2024-12-29']

In [ ]:
def each_year(start_year, end_year):
    all_sundays = []
    for year in range(start_year, end_year + 1):
        sundays = list(all_sundays_of_year(year))
        for sunday in sundays:
            all_sundays.append(sunday.strftime("%Y-%m-%d"))
    return all_sundays

each_year(2018, 2025)
#first: 2017-05-14

['2018-01-07',
 '2018-01-14',
 '2018-01-21',
 '2018-01-28',
 '2018-02-04',
 '2018-02-11',
 '2018-02-18',
 '2018-02-25',
 '2018-03-04',
 '2018-03-11',
 '2018-03-18',
 '2018-03-25',
 '2018-04-01',
 '2018-04-08',
 '2018-04-15',
 '2018-04-22',
 '2018-04-29',
 '2018-05-06',
 '2018-05-13',
 '2018-05-20',
 '2018-05-27',
 '2018-06-03',
 '2018-06-10',
 '2018-06-17',
 '2018-06-24',
 '2018-07-01',
 '2018-07-08',
 '2018-07-15',
 '2018-07-22',
 '2018-07-29',
 '2018-08-05',
 '2018-08-12',
 '2018-08-19',
 '2018-08-26',
 '2018-09-02',
 '2018-09-09',
 '2018-09-16',
 '2018-09-23',
 '2018-09-30',
 '2018-10-07',
 '2018-10-14',
 '2018-10-21',
 '2018-10-28',
 '2018-11-04',
 '2018-11-11',
 '2018-11-18',
 '2018-11-25',
 '2018-12-02',
 '2018-12-09',
 '2018-12-16',
 '2018-12-23',
 '2018-12-30',
 '2019-01-06',
 '2019-01-13',
 '2019-01-20',
 '2019-01-27',
 '2019-02-03',
 '2019-02-10',
 '2019-02-17',
 '2019-02-24',
 '2019-03-03',
 '2019-03-10',
 '2019-03-17',
 '2019-03-24',
 '2019-03-31',
 '2019-04-07',
 '2019-04-

<class 'bs4.element.ResultSet'>


NoneType